# Network Packet Sniffer
**Author:** Nisha  
**Project:** Network Packet Sniffer  
**Description:** A terminal-based local network packet sniffer built with Python and Scapy.  
Supports: Ethernet, ARP, IPv4, IPv6, ICMP/v6, TCP, UDP, DNS, HTTP

## Step 1 — Install Dependencies

In [ ]:
# Run this cell once to install required packages
!pip install scapy colorama

## Step 2 — Imports

In [ ]:
from __future__ import annotations

import argparse
import datetime
import json
import os
import signal
import sys
import threading
from collections import defaultdict
from dataclasses import asdict, dataclass, field
from typing import Callable, Optional, TextIO

from colorama import Fore, Style, init as colorama_init
from scapy.all import (
    ARP, DNS, DNSQR, ICMP, ICMPv6EchoReply, ICMPv6EchoRequest,
    IP, IPv6, Packet, Raw, TCP, UDP, Ether, conf, get_if_list, sniff,
)

colorama_init(autoreset=True)
print('All imports successful!')

## Step 3 — Data Model

In [ ]:
@dataclass
class LayerInfo:
    """Holds the name and parsed fields of one protocol layer."""
    name: str
    fields: dict = field(default_factory=dict)


@dataclass
class PacketInfo:
    """Structured representation of a single captured packet."""
    index: int
    timestamp: str
    length: int
    layers: list = field(default_factory=list)
    summary: str = ""
    src_ip: str = ""
    dst_ip: str = ""
    src_mac: str = ""
    dst_mac: str = ""
    protocol: str = ""
    src_port: Optional[int] = None
    dst_port: Optional[int] = None
    info: str = ""

print('Data model defined!')

## Step 4 — Packet Parser

In [ ]:
class PacketParser:
    """Converts raw Scapy packets into structured PacketInfo objects."""

    _counter: int = 0

    @classmethod
    def reset_counter(cls) -> None:
        cls._counter = 0

    @classmethod
    def parse(cls, packet: Packet) -> PacketInfo:
        cls._counter += 1
        ts = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
        pkt_info = PacketInfo(
            index=cls._counter,
            timestamp=ts,
            length=len(packet),
            summary=packet.summary(),
        )
        cls._parse_ethernet(packet, pkt_info)
        cls._parse_arp(packet, pkt_info)
        cls._parse_ip(packet, pkt_info)
        cls._parse_ipv6(packet, pkt_info)
        cls._parse_icmp(packet, pkt_info)
        cls._parse_tcp(packet, pkt_info)
        cls._parse_udp(packet, pkt_info)
        cls._parse_dns(packet, pkt_info)
        cls._parse_raw(packet, pkt_info)
        return pkt_info

    @staticmethod
    def _parse_ethernet(packet, info):
        if packet.haslayer(Ether):
            eth = packet[Ether]
            info.src_mac = eth.src
            info.dst_mac = eth.dst
            info.layers.append(LayerInfo("Ethernet", {"src": eth.src, "dst": eth.dst, "type": hex(eth.type)}))

    @staticmethod
    def _parse_arp(packet, info):
        if packet.haslayer(ARP):
            arp = packet[ARP]
            op = "Request" if arp.op == 1 else "Reply"
            info.protocol = "ARP"
            info.src_ip   = arp.psrc
            info.dst_ip   = arp.pdst
            info.info     = f"ARP {op}: {arp.psrc} -> {arp.pdst}"
            info.layers.append(LayerInfo("ARP", {
                "operation": op, "sender_mac": arp.hwsrc,
                "sender_ip": arp.psrc, "target_mac": arp.hwdst, "target_ip": arp.pdst,
            }))

    @staticmethod
    def _parse_ip(packet, info):
        if packet.haslayer(IP):
            ip = packet[IP]
            info.src_ip = ip.src
            info.dst_ip = ip.dst
            if not info.protocol:
                info.protocol = "IP"
            info.layers.append(LayerInfo("IPv4", {
                "src": ip.src, "dst": ip.dst, "ttl": ip.ttl,
                "proto": ip.proto, "id": ip.id, "flags": str(ip.flags), "tos": ip.tos,
            }))

    @staticmethod
    def _parse_ipv6(packet, info):
        if packet.haslayer(IPv6):
            ip6 = packet[IPv6]
            info.src_ip = ip6.src
            info.dst_ip = ip6.dst
            if not info.protocol:
                info.protocol = "IPv6"
            info.layers.append(LayerInfo("IPv6", {
                "src": ip6.src, "dst": ip6.dst, "hop_limit": ip6.hlim, "next_header": ip6.nh,
            }))

    @staticmethod
    def _parse_icmp(packet, info):
        if packet.haslayer(ICMP):
            icmp = packet[ICMP]
            icmp_types = {0: "Echo Reply", 8: "Echo Request", 3: "Dest Unreachable", 11: "Time Exceeded", 5: "Redirect"}
            type_name = icmp_types.get(icmp.type, f"Type {icmp.type}")
            info.protocol = "ICMP"
            info.info = f"ICMP {type_name}"
            info.layers.append(LayerInfo("ICMP", {"type": type_name, "code": icmp.code}))
        elif packet.haslayer(ICMPv6EchoRequest):
            info.protocol = "ICMPv6"
            info.info = "ICMPv6 Echo Request"
            info.layers.append(LayerInfo("ICMPv6", {"type": "Echo Request"}))
        elif packet.haslayer(ICMPv6EchoReply):
            info.protocol = "ICMPv6"
            info.info = "ICMPv6 Echo Reply"
            info.layers.append(LayerInfo("ICMPv6", {"type": "Echo Reply"}))

    @staticmethod
    def _tcp_flags(flags) -> str:
        flag_map = {"F": "FIN", "S": "SYN", "R": "RST", "P": "PSH", "A": "ACK", "U": "URG", "E": "ECE", "C": "CWR"}
        active = [long for short, long in flag_map.items() if short in str(flags)]
        return ",".join(active) if active else str(flags)

    @staticmethod
    def _parse_tcp(packet, info):
        if packet.haslayer(TCP):
            tcp = packet[TCP]
            flags = PacketParser._tcp_flags(tcp.flags)
            info.protocol = "TCP"
            info.src_port = tcp.sport
            info.dst_port = tcp.dport
            info.info = f"{tcp.sport} -> {tcp.dport} [{flags}] Seq={tcp.seq} Ack={tcp.ack} Win={tcp.window}"
            info.layers.append(LayerInfo("TCP", {
                "src_port": tcp.sport, "dst_port": tcp.dport,
                "flags": flags, "seq": tcp.seq, "ack": tcp.ack, "window": tcp.window,
            }))

    @staticmethod
    def _parse_udp(packet, info):
        if packet.haslayer(UDP):
            udp = packet[UDP]
            if not info.protocol or info.protocol == "IP":
                info.protocol = "UDP"
            info.src_port = udp.sport
            info.dst_port = udp.dport
            if not info.info:
                info.info = f"{udp.sport} -> {udp.dport} Len={udp.len}"
            info.layers.append(LayerInfo("UDP", {
                "src_port": udp.sport, "dst_port": udp.dport,
                "length": udp.len, "checksum": hex(udp.chksum) if udp.chksum else "-",
            }))

    @staticmethod
    def _parse_dns(packet, info):
        if packet.haslayer(DNS):
            dns = packet[DNS]
            info.protocol = "DNS"
            if dns.qr == 0:
                qname = dns[DNSQR].qname.decode(errors="replace") if dns.qdcount > 0 else ""
                info.info = f"DNS Query: {qname}"
                info.layers.append(LayerInfo("DNS", {"type": "Query", "id": dns.id, "question": qname}))
            else:
                answers = []
                ans = dns.an
                while ans and hasattr(ans, "rdata"):
                    try:
                        answers.append(str(ans.rdata))
                    except Exception:
                        pass
                    ans = ans.payload if hasattr(ans, "payload") else None
                info.info = f"DNS Response: {', '.join(answers[:3])}"
                info.layers.append(LayerInfo("DNS", {"type": "Response", "id": dns.id, "answers": answers}))

    @staticmethod
    def _parse_raw(packet, info):
        if packet.haslayer(Raw):
            raw_data = bytes(packet[Raw])
            preview  = raw_data[:64]
            try:
                text = preview.decode("utf-8", errors="ignore")
                if text.startswith(("GET ", "POST ", "PUT ", "DELETE ", "HTTP/")):
                    first_line = text.split("\r\n")[0]
                    info.protocol = "HTTP"
                    info.info     = first_line
                    info.layers.append(LayerInfo("HTTP", {"first_line": first_line}))
                    return
            except Exception:
                pass
            info.layers.append(LayerInfo("Raw", {"length": len(raw_data), "preview": preview.hex()}))

print('PacketParser defined!')

## Step 5 — Packet Capture Engine

In [ ]:
class PacketCapture:
    """Captures packets from a network interface using Scapy."""

    def __init__(self, interface=None, bpf_filter=None, packet_count=0, on_packet=None):
        self.interface    = interface or conf.iface
        self.bpf_filter   = bpf_filter
        self.packet_count = packet_count
        self.on_packet    = on_packet
        self._stop_event  = threading.Event()
        self._thread      = None
        self.captured     = []

    @staticmethod
    def list_interfaces():
        return get_if_list()

    def start(self, blocking=False):
        self._stop_event.clear()
        if blocking:
            self._run()
        else:
            self._thread = threading.Thread(target=self._run, daemon=True)
            self._thread.start()

    def stop(self):
        self._stop_event.set()
        if self._thread and self._thread.is_alive():
            self._thread.join(timeout=3)

    def _run(self):
        sniff(
            iface=self.interface,
            filter=self.bpf_filter if self.bpf_filter else None,
            count=self.packet_count,
            prn=self._handle_packet,
            store=False,
            stop_filter=lambda _: self._stop_event.is_set(),
        )

    def _handle_packet(self, packet):
        self.captured.append(packet)
        if self.on_packet:
            self.on_packet(packet)

print('PacketCapture defined!')

## Step 6 — Display (Terminal UI)

In [ ]:
PROTO_COLORS = {
    "TCP": Fore.CYAN, "UDP": Fore.GREEN, "DNS": Fore.YELLOW,
    "HTTP": Fore.MAGENTA, "ICMP": Fore.BLUE, "ICMPv6": Fore.BLUE,
    "ARP": Fore.WHITE, "IPv6": Fore.LIGHTCYAN_EX, "IP": Fore.LIGHTWHITE_EX,
}
HEADER_COLOR  = Fore.LIGHTWHITE_EX + Style.BRIGHT
SECTION_COLOR = Fore.LIGHTYELLOW_EX + Style.BRIGHT
FIELD_COLOR   = Fore.LIGHTBLACK_EX


class Display:
    """Handles all terminal output and optional log-file writing."""

    _COL = {"no": 5, "time": 12, "proto": 7, "src": 22, "dst": 22, "len": 6}

    def __init__(self, verbose=False, log_file=None):
        self.verbose  = verbose
        self.log_file = log_file
        self._log_fh      = None
        self._log_json    = False
        self._log_records = []
        self._stats        = defaultdict(int)
        self._total_bytes  = 0
        self._packet_count = 0
        self._terminal_width = self._get_terminal_width()
        self._open_log_file()

    def print_header(self):
        w = self._terminal_width
        print(HEADER_COLOR + "=" * w)
        print(f"{'NETWORK PACKET SNIFFER':^{w}}")
        print("=" * w + Style.RESET_ALL)
        self._print_table_row(no="No.", time="Time", proto="Proto",
                               src="Source", dst="Destination", length="Len", info="Info", color=HEADER_COLOR)
        print(HEADER_COLOR + "-" * w + Style.RESET_ALL)

    def display(self, pkt_info):
        self._packet_count += 1
        self._total_bytes  += pkt_info.length
        self._stats[pkt_info.protocol or "OTHER"] += 1
        color = PROTO_COLORS.get(pkt_info.protocol, Fore.RESET)
        info_width = max(10, self._terminal_width - sum(self._COL.values()) - 14)
        info_str = (pkt_info.info or pkt_info.summary)[:info_width]
        self._print_table_row(
            no=str(pkt_info.index), time=pkt_info.timestamp,
            proto=pkt_info.protocol or "?",
            src=self._format_endpoint(pkt_info.src_ip, pkt_info.src_port),
            dst=self._format_endpoint(pkt_info.dst_ip, pkt_info.dst_port),
            length=str(pkt_info.length), info=info_str, color=color,
        )
        if self.verbose:
            self.print_detail(pkt_info)
        self._write_log(pkt_info)

    def print_detail(self, pkt_info):
        w = self._terminal_width
        print(SECTION_COLOR + f"  +-- Packet #{pkt_info.index} detail " + "-" * max(0, w - 24) + Style.RESET_ALL)
        for layer in pkt_info.layers:
            print(SECTION_COLOR + f"  |  [{layer.name}]" + Style.RESET_ALL)
            for k, v in layer.fields.items():
                print(f"  |    {FIELD_COLOR}{k:<18}{Style.RESET_ALL}{v}")
        print(SECTION_COLOR + "  +" + "-" * (w - 4) + Style.RESET_ALL)

    def print_stats(self):
        w = self._terminal_width
        print()
        print(HEADER_COLOR + "=" * w)
        print(f"{'SESSION STATISTICS':^{w}}")
        print("=" * w + Style.RESET_ALL)
        print(f"  {'Total packets':<30} {self._packet_count}")
        print(f"  {'Total bytes captured':<30} {self._total_bytes:,}")
        print()
        print(SECTION_COLOR + "  Protocol breakdown:" + Style.RESET_ALL)
        for proto, count in sorted(self._stats.items(), key=lambda x: -x[1]):
            bar_len = int(count / max(self._packet_count, 1) * 30)
            bar   = "#" * bar_len
            color = PROTO_COLORS.get(proto, Fore.RESET)
            print(f"  {color}{proto:<10}{Style.RESET_ALL}  {count:>5}  {color}{bar}{Style.RESET_ALL}")
        print(HEADER_COLOR + "=" * w + Style.RESET_ALL)

    def close(self):
        if self._log_fh:
            if self._log_json:
                json.dump(self._log_records, self._log_fh, indent=2)
            self._log_fh.close()
            self._log_fh = None

    def _print_table_row(self, *, no, time, proto, src, dst, length, info, color=""):
        c = self._COL
        row = (f"{no:>{c['no']}}  {time:<{c['time']}}  {proto:<{c['proto']}}  "
               f"{src:<{c['src']}}  {dst:<{c['dst']}}  {length:>{c['len']}}  {info}")
        print(color + row + Style.RESET_ALL)

    @staticmethod
    def _format_endpoint(ip, port):
        if ip and port:
            return f"{ip}:{port}"
        return ip or "-"

    @staticmethod
    def _get_terminal_width():
        try:
            return min(os.get_terminal_size().columns, 200)
        except OSError:
            return 120

    def _open_log_file(self):
        if not self.log_file:
            return
        self._log_json = self.log_file.endswith(".json")
        self._log_fh   = open(self.log_file, "w", encoding="utf-8")
        if not self._log_json:
            self._log_fh.write("Network Packet Sniffer — Capture Log\n")
            self._log_fh.write("=" * 80 + "\n")

    def _write_log(self, pkt_info):
        if not self._log_fh:
            return
        if self._log_json:
            self._log_records.append(asdict(pkt_info))
        else:
            self._log_fh.write(
                f"[{pkt_info.index}] {pkt_info.timestamp}  "
                f"{pkt_info.protocol or '?':<7}  "
                f"{self._format_endpoint(pkt_info.src_ip, pkt_info.src_port):<25} -> "
                f"{self._format_endpoint(pkt_info.dst_ip, pkt_info.dst_port):<25}  "
                f"{pkt_info.length}B  {pkt_info.info or pkt_info.summary}\n"
            )
            self._log_fh.flush()

print('Display defined!')

## Step 7 — List Available Network Interfaces

In [ ]:
# Run this cell to see all available interfaces on your machine
print("Available network interfaces:")
for idx, iface in enumerate(PacketCapture.list_interfaces(), 1):
    print(f"  [{idx}] {iface}")

## Step 8 — Run the Sniffer

> ⚠️ **Requires Administrator / sudo privileges**  
> Change `INTERFACE` to one of the interfaces listed above.  
> Change `COUNT` to how many packets you want to capture.  
> Press the **Stop (■)** button in Jupyter to stop an unlimited capture.

In [ ]:
# ── Configuration ──────────────────────────────────────────
INTERFACE = None          # e.g. "Wi-Fi" or "eth0"  (None = default)
BPF_FILTER = None         # e.g. "tcp", "udp port 53"  (None = all traffic)
COUNT = 20                # number of packets to capture (0 = unlimited)
VERBOSE = True            # True = show layer details per packet
LOG_FILE = None           # e.g. "capture.json" or "capture.txt"  (None = no log)
# ───────────────────────────────────────────────────────────

PacketParser.reset_counter()
display = Display(verbose=VERBOSE, log_file=LOG_FILE)

def on_packet(packet):
    pkt_info = PacketParser.parse(packet)
    display.display(pkt_info)

capture = PacketCapture(
    interface=INTERFACE,
    bpf_filter=BPF_FILTER,
    packet_count=COUNT,
    on_packet=on_packet,
)

display.print_header()
capture.start(blocking=True)
display.print_stats()
display.close()

## Step 9 — Capture DNS Traffic Only

In [ ]:
PacketParser.reset_counter()
display_dns = Display(verbose=False)

capture_dns = PacketCapture(
    bpf_filter="udp port 53",
    packet_count=10,
    on_packet=lambda p: display_dns.display(PacketParser.parse(p)),
)

display_dns.print_header()
capture_dns.start(blocking=True)
display_dns.print_stats()
display_dns.close()

## Step 10 — Capture TCP Traffic Only

In [ ]:
PacketParser.reset_counter()
display_tcp = Display(verbose=False)

capture_tcp = PacketCapture(
    bpf_filter="tcp",
    packet_count=10,
    on_packet=lambda p: display_tcp.display(PacketParser.parse(p)),
)

display_tcp.print_header()
capture_tcp.start(blocking=True)
display_tcp.print_stats()
display_tcp.close()